# Auditoría del Laboratorio 3: CNN2 con data augmentation

**Proyecto 3 — Optimización de reconocimiento ASL con CUDA**

Esta auditoría reconstruye el flujo del Lab3 original antes de migrarlo de TensorFlow/Keras a PyTorch. Se revisaron los notebooks, scripts, README, métricas, resultados y modelos guardados dentro de `original_lab3/`. No se entrena ningún modelo, no se descarga ningún dataset y no se modifica el laboratorio original.

Las cantidades históricas se reportan solamente cuando están respaldadas por salidas o artefactos del Lab3. El código de este notebook verifica archivos y metadatos sin presentar esos resultados como una nueva ejecución.

In [ ]:
from pathlib import Path
import json
import math
import zipfile

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
LAB = ROOT / 'original_lab3'
NB_DIR = LAB / 'Notebooks'
RESULTS = NB_DIR / 'results'

required_sources = [
    LAB / 'README.md',
    LAB / 'requirements.txt',
    LAB / 'src' / 'prepare_asl_data.py',
    LAB / 'Laboratorio_3_EDA_ASL.ipynb',
    NB_DIR / '0.Laboratorio_3_EDA_ASL.ipynb',
    NB_DIR / '01_modelos_dl.ipynb',
    NB_DIR / '02_random_forest_comparacion_final.ipynb',
]
assert LAB.is_dir(), f'No se encontró {LAB}'
for path in required_sources:
    assert path.is_file(), f'Falta la fuente esperada: {path.relative_to(ROOT)}'
print('Raíz:', ROOT)
print('Fuentes principales verificadas:', len(required_sources))

## 1. Fuentes auditadas y flujo completo

| Etapa | Implementación original | Entrada → salida | Cantidad respaldada |
|---|---|---|---:|
| Inventario | EDA y `src/prepare_asl_data.py` | carpetas por clase → rutas y etiquetas | 29 × 3,000 = 87,000 imágenes |
| Selección | bloques consecutivos de 30; semilla 42 | 100 bloques/clase → 20 bloques/clase | 600 por clase; 17,400 total |
| Partición | 14/3/3 bloques por clase | muestra → train/val/test | 12,180 / 2,610 / 2,610 |
| Decodificación | Pillow | JPEG/PNG → RGB | una vez por imagen materializada |
| Preprocesamiento | EXIF, RGB, resize bilineal | 200×200×3 → 64×64×3 `uint8` | 17,400 imágenes |
| Persistencia | NumPy | arreglos → seis `.npy` + `classes.json` | artefactos ausentes actualmente |
| Carga | `np.load` | `.npy` → memoria | formas verificadas en salida histórica |
| Normalización | `astype(float32) / 255.0` | `[0,255]` → `[0,1]` | todos los píxeles de los 3 splits |
| Augmentation | capa Keras al inicio de `CNN2_aug` | lote train → lote transformado | solo durante `training=True` |
| Entrenamiento | `model.fit` | train + validación → pesos | máximo 30 épocas; early stopping |
| Evaluación | `evaluate`, `predict`, sklearn | val/test → accuracy y F1 macro | prueba interna de 2,610 imágenes |
| Exportación | `model.save` y JSON/CSV | modelo/métricas → artefactos | `mejor_modelo_aug.keras` disponible |

La selección se realiza **antes** del augmentation. Cada bloque reúne archivos con números de secuencia consecutivos; ningún bloque ni ruta se comparte entre splits según la salida guardada del EDA. Esto reduce fuga por fotografías contiguas, pero no garantiza separación por persona o sesión porque el dataset no aporta esos identificadores.

### Versiones e inconsistencias encontradas

- `original_lab3/Laboratorio_3_EDA_ASL.ipynb` es una versión anterior de 36 celdas centrada en EDA; no contiene la partición ni la exportación.
- `original_lab3/Notebooks/0.Laboratorio_3_EDA_ASL.ipynb` tiene 45 celdas y añade el split por bloques, el pipeline final y la exportación consumida por `01_modelos_dl.ipynb`; por eso es la fuente operativa para la migración.
- El README original todavía usa rutas `Lab3/...` y enlaza `INFORME_FINAL_LAB3.md`, archivo que no existe con ese nombre en el árbol actual. El informe disponible es `Informe_Laboratorio_3.md`.
- `src/build_final_notebook.py` también espera `INFORME_FINAL_LAB3.md`; por tanto, ese generador no es reproducible sin corregir la ruta/nombre en una etapa futura. No se modifica aquí por la restricción de conservar el Lab3.
- `src/prepare_asl_data.py` sí respalda los valores centrales del EDA (semilla 42, 64×64, bloques 14/3/3, 29 clases y 17,400 imágenes), pero los datos necesarios para ejecutarlo faltan.

In [ ]:
N_CLASSES = 29
BLOCK_SIZE = 30
BLOCKS = {'train': 14, 'val': 3, 'test': 3}
N = {name: N_CLASSES * count * BLOCK_SIZE for name, count in BLOCKS.items()}
assert N == {'train': 12180, 'val': 2610, 'test': 2610}
assert sum(N.values()) == 17400

PIXELS_PER_IMAGE = 64 * 64 * 3
normalized_values = {name: count * PIXELS_PER_IMAGE for name, count in N.items()}
print('Imágenes por split:', N)
print('Valores float32 normalizados por split:', normalized_values)
print('Total de divisiones por 255:', sum(normalized_values.values()))

## 2. Arquitectura exacta de CNN2 con augmentation

Entrada Keras: `(batch, 64, 64, 3)` (NHWC). Las convoluciones usan bias, kernel 3×3, stride 1 y `padding='same'`; los MaxPool usan 2×2, stride 2 y padding válido.

| # | Capa | Configuración | Salida sin batch | Parámetros |
|---:|---|---|---:|---:|
| 0 | Input | RGB `float32` | 64×64×3 | 0 |
| 1 | Augmentation | 5 transformaciones, solo train | 64×64×3 | 0 |
| 2 | Conv2D | 32 filtros, 3×3, same, ReLU | 64×64×32 | 896 |
| 3 | BatchNormalization | eje canales, ε=0.001, momentum=0.99 | 64×64×32 | 128 |
| 4 | MaxPooling2D | 2×2, stride 2 | 32×32×32 | 0 |
| 5 | Dropout | p=0.25 | 32×32×32 | 0 |
| 6 | Conv2D | 64 filtros, 3×3, same, ReLU | 32×32×64 | 18,496 |
| 7 | BatchNormalization | ε=0.001, momentum=0.99 | 32×32×64 | 256 |
| 8 | MaxPooling2D | 2×2, stride 2 | 16×16×64 | 0 |
| 9 | Dropout | p=0.25 | 16×16×64 | 0 |
| 10 | Conv2D | 128 filtros, 3×3, same, ReLU | 16×16×128 | 73,856 |
| 11 | BatchNormalization | ε=0.001, momentum=0.99 | 16×16×128 | 512 |
| 12 | MaxPooling2D | 2×2, stride 2 | 8×8×128 | 0 |
| 13 | Dropout | p=0.25 | 8×8×128 | 0 |
| 14 | GlobalAveragePooling2D | promedio sobre 8×8 | 128 | 0 |
| 15 | Dropout | p=0.50 | 128 | 0 |
| 16 | Dense | 29 unidades, softmax | 29 | 3,741 |

**Total verificado:** 97,885 parámetros: 97,437 entrenables y 448 no entrenables (medias y varianzas móviles de BatchNorm).

**Entrenamiento original:** Adam (`lr=0.001`, β₁=0.9, β₂=0.999, ε=1e-7, sin weight decay), `sparse_categorical_crossentropy`, accuracy, batch 32, hasta 30 épocas y `EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)`. La ejecución histórica de `CNN2_aug` terminó tras 18 épocas y restauró los pesos de la mejor pérdida de validación (época 13).

In [ ]:
model_path = NB_DIR / 'mejor_modelo_aug.keras'
metrics_path = RESULTS / 'metrics_CNN2_aug.json'
assert model_path.is_file() and metrics_path.is_file()

with zipfile.ZipFile(model_path) as archive:
    config = json.loads(archive.read('config.json'))
top_layers = config['config']['layers']
layer_names = [layer['class_name'] for layer in top_layers]
expected = [
    'InputLayer', 'Sequential', 'Conv2D', 'BatchNormalization',
    'MaxPooling2D', 'Dropout', 'Conv2D', 'BatchNormalization',
    'MaxPooling2D', 'Dropout', 'Conv2D', 'BatchNormalization',
    'MaxPooling2D', 'Dropout', 'GlobalAveragePooling2D', 'Dropout', 'Dense'
]
assert layer_names == expected
assert config['compile_config']['loss'] == 'sparse_categorical_crossentropy'

metrics = json.loads(metrics_path.read_text(encoding='utf-8'))
assert metrics['modelo'] == 'CNN2_aug' and metrics['params'] == 97885
print('Secuencia de capas verificada:', ' → '.join(layer_names))
print('Métricas históricas leídas (no recalculadas):', metrics)

## 3. Augmentation exacto

| Orden | Operación Keras | Rango/configuración guardada | Observación para paridad |
|---:|---|---|---|
| 1 | `RandomRotation(0.04)` | −0.04 a +0.04 vueltas = ±14.4° | bilineal, relleno reflect |
| 2 | `RandomTranslation(0.1, 0.1)` | ±10% de alto y ancho | bilineal, relleno reflect |
| 3 | `RandomZoom(0.1)` | factor vertical −0.1 a +0.1; ancho enlazado | bilineal, relleno reflect |
| 4 | `RandomBrightness(0.2, value_range=(0,1))` | delta de brillo en ±20% del rango | coherente con entrada normalizada |
| 5 | `RandomContrast(0.2)` | factor alrededor de 1 ± 0.2 | el modelo guardado conserva `value_range=(0,255)` |

No hay flip horizontal. La decisión original es correcta para esta auditoría: invertir una mano puede alterar la orientación semántica de la seña. Las capas tienen `seed=None`, pero el notebook llama `tf.keras.utils.set_random_seed(42)` antes de construirlas.

> **Hallazgo de paridad:** las imágenes se normalizan a `[0,1]`, mientras `RandomContrast` conserva el rango predeterminado `[0,255]`. No debe corregirse silenciosamente durante la migración. Primero se debe decidir entre reproducir el comportamiento guardado o corregirlo y declarar que el experimento cambia.

## 4. Operaciones repetitivas

Sea `E` el número real de épocas y `B` el batch size. Para el original, `B=32`; `ceil(12,180/32)=381` batches de entrenamiento y `ceil(2,610/32)=82` batches de validación/prueba.

| Operación | Frecuencia/fórmula | CNN2_aug histórica (`E=18`, `B=32`) |
|---|---:|---:|
| Presentaciones train al augmentation | `E × 12,180` imágenes | 219,240 |
| Secuencias de 5 transformaciones | `E × 12,180` | 219,240 secuencias |
| Pasos forward + backward + Adam | `E × ceil(12,180/B)` | 6,858 pasos |
| Forward de validación durante fit | `E × 2,610` imágenes | 46,980 |
| Batches de validación durante fit | `E × ceil(2,610/B)` | 1,476 batches |
| Evaluación final en validación | `ceil(2,610/B)` | 82 batches |
| Evaluación final en test | `ceil(2,610/B)` | 82 batches |
| Predicción final en test | `ceil(2,610/B)` | 82 batches |
| Actualización de BatchNorm | una por batch train y por capa BN | 6,858 × 3 |
| Cálculo de F1 macro | una vez sobre 2,610 predicciones | 1 |

Las 18 épocas provienen del log histórico; una ejecución nueva puede detenerse en otra época. El mejor checkpoint lógico fue el de la época 13, pero se ejecutaron cinco épocas posteriores por la paciencia configurada.

In [ ]:
def repeated_ops(epochs, batch_size=32):
    train_batches = math.ceil(N['train'] / batch_size)
    eval_batches = math.ceil(N['val'] / batch_size)
    return {
        'train_image_presentations': epochs * N['train'],
        'optimizer_steps': epochs * train_batches,
        'validation_image_presentations_during_fit': epochs * N['val'],
        'validation_batches_during_fit': epochs * eval_batches,
        'batches_per_train_epoch': train_batches,
        'batches_per_eval_pass': eval_batches,
    }

historical = repeated_ops(epochs=18, batch_size=32)
assert historical['optimizer_steps'] == 6858
print(historical)

## 5. Asignación CPU y GPU/CUDA

| Operación | Debe quedar en CPU | Candidata a GPU/CUDA | Motivo |
|---|:---:|:---:|---|
| Recorrer carpetas, validar rutas y leer JSON/CSV | ✓ |  | I/O y control; poco paralelismo numérico |
| Decodificar JPEG/PNG y corregir EXIF | ✓ |  | Pillow y metadatos trabajan en CPU |
| Seleccionar bloques/splits y mapear etiquetas | ✓ |  | operación pequeña y reproducible |
| Resize al materializar `.npy` | ✓ | opcional | mantener CPU para reproducir el pipeline original |
| Cargar batches y workers del DataLoader | ✓ |  | preparación y transferencia de datos |
| Normalización | posible | ✓ | puede fusionarse en el pipeline tensorial |
| Augmentation geométrico/fotométrico | posible | ✓ | repetido por imagen y época; medir ambas opciones |
| Conv, ReLU, pooling, BatchNorm, dropout, GAP, Dense |  | ✓ | núcleo intensivo y altamente paralelo |
| Loss, backward y Adam |  | ✓ | deben permanecer junto a parámetros/activaciones |
| Acumular predicciones y calcular F1/confusión | ✓ | posible | sklearn exige CPU; transferir una vez al final |
| Guardar pesos, métricas y figuras | ✓ |  | I/O |

La transferencia host→device debe hacerse por batch, idealmente con memoria pinned y copias no bloqueantes si el hardware lo permite. El benchmark futuro debe separar tiempo de carga/preprocesamiento, transferencia y cómputo para no atribuir todo a CUDA.

## 6. Artefactos disponibles y faltantes

| Artefacto | Estado actual | Consecuencia |
|---|---|---|
| 4 notebooks del Lab3 (2 versiones EDA, modelos y comparación) | disponibles | permite reconstruir el flujo y leer salidas históricas |
| scripts de preparación y Random Forest | disponibles | segunda fuente reproducible para selección/preprocesamiento |
| generador `build_final_notebook.py` | disponible pero con entrada faltante | apunta a `INFORME_FINAL_LAB3.md`, nombre no presente |
| `mejor_modelo.keras` y `mejor_modelo_aug.keras` | disponibles | arquitectura, configuración y pesos Keras auditables |
| métricas de 6 redes y Random Forest | disponibles | referencia histórica, no benchmark PyTorch |
| CSV/PNG de comparaciones y matrices | disponibles | evidencia de evaluación original |
| Random Forest `.joblib` | disponible | artefacto auxiliar, no objetivo de la migración |
| 150 fotos propias (15 clases × 10) | disponibles | conjunto externo parcial |
| dataset ASL Alphabet de entrenamiento (87,000) | **faltante** | no se puede reconstruir selección/splits desde cero |
| seis arreglos `X/y_{train,val,test}.npy` | **faltantes** | no se puede entrenar/evaluar la CNN con la partición original |
| `classes.json` y `preprocess_meta.json` materializados | **faltantes** | el orden se infiere del notebook, no se verifica contra el archivo exportado |
| set oficial Kaggle de 28 imágenes | **faltante** | no se pueden repetir esas predicciones |
| identificadores de sujeto/sesión | no existen en la evidencia | no se puede demostrar separación por persona |
| entorno TensorFlow local | **incompatible actualmente** | TF 2.10.1/Keras 2.10.0 no importan con NumPy 2.2.6; además falta `cudart64_110.dll` |

Orden de clases documentado por la salida del EDA: `a,b,c,d,del,e,f,g,h,i,j,k,l,m,n,nothing,o,p,q,r,s,space,t,u,v,w,x,y,z`. Debe conservarse exactamente al migrar.

In [ ]:
training_candidates = [
    LAB / 'data' / 'asl_alphabet_train',
    LAB / 'data' / 'asl_alphabet_train' / 'asl_alphabet_train',
]
npy_expected = [NB_DIR / 'data' / f'{kind}_{split}.npy'
                for split in ('train', 'val', 'test')
                for kind in ('X', 'y')]
own_photos = [p for p in (LAB / 'asl_alphabet_test').rglob('*') if p.is_file()]
status = {
    'dataset_train_disponible': any(path.is_dir() for path in training_candidates),
    'npy_presentes': sum(path.is_file() for path in npy_expected),
    'npy_esperados': len(npy_expected),
    'fotos_propias': len(own_photos),
    'modelos_keras': len(list(NB_DIR.glob('*.keras'))),
    'metricas_json': len(list(RESULTS.glob('metrics_*.json'))),
}
assert status['fotos_propias'] == 150
print(status)

## 7. Diferencias críticas Keras → PyTorch

| Tema | Keras original | Decisión necesaria en PyTorch |
|---|---|---|
| Layout | NHWC | convertir a NCHW una sola vez y verificar formas |
| Normalización | NumPy `float32 / 255` antes de `fit` | evitar dividir dos veces; ubicarla en Dataset o modelo |
| `same` | Conv 3×3 conserva alto/ancho | `padding=1` con stride 1 |
| BatchNorm ε | 0.001 | `eps=1e-3`, no el default 1e-5 |
| Momentum BN | Keras 0.99 pondera el histórico | PyTorch usa convención inversa; `momentum=0.01` aproxima la actualización |
| Varianza BN | implementación Keras/TF | validar running variance; puede impedir igualdad bit a bit |
| Salida/loss | softmax + sparse categorical CE | `CrossEntropyLoss` espera logits: omitir softmax durante entrenamiento |
| Adam ε | 1e-7 | configurar 1e-7; PyTorch usa 1e-8 por defecto |
| Inicialización | Glorot uniform en Conv/Dense | definirla explícitamente si se compara entrenamiento desde cero |
| Pesos Conv | HWIO | PyTorch OIHW; transponer al convertir pesos |
| Pesos Dense | entrada×salida | PyTorch salida×entrada; transponer |
| Dropout/augmentation | dependen de `training` | alternar correctamente `train()`/`eval()` |
| Augmentation | capas Keras, reflect/bilineal | igualar grados, traslación, zoom, brillo, contraste, orden y relleno |
| RNG | semilla global 42, capas con seed nula | sembrar Python/NumPy/PyTorch/workers; aun así CUDA puede no ser idéntico |
| Etiquetas | 29 índices en orden lexicográfico documentado | congelar el mismo mapa y guardarlo con checkpoints |
| Early stopping | restaura mejor `val_loss` | guardar/restaurar `state_dict` del mejor epoch |

Antes de optimizar CUDA conviene hacer una prueba de paridad con un lote fijo: mismas imágenes, augmentation desactivado y comparación de shapes/logits entre el `.keras` y PyTorch. Después se valida la conversión de pesos; solo entonces se mide rendimiento.

## 8. Conclusiones y pendientes

La arquitectura objetivo queda identificada y respaldada por dos fuentes: el notebook original y `mejor_modelo_aug.keras`. El pipeline usa una submuestra balanceada de 17,400 imágenes, partición por bloques 70/15/15, resize bilineal a 64×64, normalización `[0,1]` y augmentation en línea solo sobre train.

Para iniciar la implementación PyTorch faltan estas decisiones o insumos:

1. recuperar legítimamente el dataset de entrenamiento o los seis `.npy` originales, sin descargarlos como parte de esta auditoría;
2. recuperar `classes.json`/metadatos y verificar hashes o reproducir exactamente el split con semilla 42;
3. decidir si se replica el `value_range=(0,255)` de RandomContrast o se corrige con un experimento declarado;
4. definir si la primera meta es paridad de inferencia mediante conversión de pesos o repetición del entrenamiento desde cero;
5. crear un entorno aislado y versionado para cualquier validación Keras: la ejecución histórica usó TensorFlow 2.21.0, mientras el entorno global actual tiene TF 2.10.1/Keras 2.10.0 con NumPy 2.2.6 y falla por incompatibilidad binaria;
6. fijar protocolo de benchmark (warm-up, sincronización CUDA, repeticiones, batch sizes, memoria y tiempos por etapa).

Las métricas históricas (`accuracy_test=0.8490`, `F1 macro=0.8486`, 664.6 s) son evidencia del Lab3 original ejecutado en CPU según su log. No son resultados del Proyecto 3 ni una línea base CUDA hasta repetir el protocolo en el entorno actual.